# Week 4: Attention

Attention may be the most important idea in transformers, and it's a [much broader idea](https://en.wikipedia.org/wiki/Attention_(machine_learning)) beyond its use in transformers alone.

Inside a transformer, attention is the one step where positions communicate. Everything else (the embedding lookup, the feed-forward layers) processes each position independently. Attention is what lets one position pull in information from the others positions, so a word's embedding can become aware of its context.

Today we'll understand attention as its own thing first, then plug it into the bigger picture of tokens and embeddings next week.

So what is attention?  
**It's a "*soft* dictionary."**

First, recall how a Python dictionary works:

```python
my_dict = {'a': 1, 'b': 2, 'c': 3}

print(my_dict['b'])  # 2
```

You store **keys** (`a`, `b`, `c`), match them against a **query** (`'b'`), and get back a **value** (`2`).

But this only works on exact matches:

```python
my_dict['bb']  # KeyError
```

Embeddings don't work that way. `cat` and `kitten` are *close* but never *equal*, so what would it mean to look something up by "approximately this"? <span color="red">And how could we train such a lookup?</span>

Let's build attention from scratch to find out!

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 1280 650" width="100%" style="max-width:1280px" font-family="system-ui, -apple-system, Segoe UI, Helvetica, Arial, sans-serif" role="img" aria-label="Diagram: a Python dictionary does exact-match lookup, while attention is a soft dictionary that blends values by similarity">
<text x="50" y="32" font-size="20" text-anchor="start" font-weight="600" fill="currentColor">From a dictionary to a soft dictionary</text>

<!-- ============ PANEL 1: Python dict ============ -->
<rect x="50" y="60" width="1010" height="200" rx="14" fill="currentColor" fill-opacity="0.025" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.5"/>
<text x="70" y="84" font-size="14" text-anchor="start" font-weight="600" fill="currentColor">Python dict: exact match</text>
<text x="115" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">query</text>
<text x="340" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">keys</text>
<text x="480" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">match?</text>
<text x="640" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">values</text>
<text x="840" y="102" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">result</text>

<!-- query -->
<rect x="70" y="154" width="90" height="32" rx="6" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.5"/>
<text x="115" y="175" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">'b'</text>

<!-- query -> keys -->
<path d="M160,170 H230 V125 H293" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-opacity="0.3" stroke-linejoin="round"/>
<polygon points="299,125 292,121.5 292,128.5" fill="#E8833A" fill-opacity="0.3"/>
<path d="M160,170 H230 V215 H293" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-opacity="0.3" stroke-linejoin="round"/>
<polygon points="299,215 292,211.5 292,218.5" fill="#E8833A" fill-opacity="0.3"/>
<path d="M160,170 H293" fill="none" stroke="#E8833A" stroke-width="2.2" stroke-linejoin="round"/>
<polygon points="299,170 292,166.5 292,173.5" fill="#E8833A"/>

<!-- keys -->
<rect x="300" y="109" width="80" height="32" rx="6" fill="#3B82F6" fill-opacity="0.16" stroke="#3B82F6" stroke-width="1.2"/>
<text x="340" y="130" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">'a'</text>
<rect x="300" y="154" width="80" height="32" rx="6" fill="#3B82F6" fill-opacity="0.16" stroke="#3B82F6" stroke-width="1.2"/>
<text x="340" y="175" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">'b'</text>
<rect x="300" y="199" width="80" height="32" rx="6" fill="#3B82F6" fill-opacity="0.16" stroke="#3B82F6" stroke-width="1.2"/>
<text x="340" y="220" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">'c'</text>

<!-- keys -> match -->
<line x1="380" y1="125" x2="464" y2="125" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.25"/>
<line x1="380" y1="170" x2="464" y2="170" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<line x1="380" y1="215" x2="464" y2="215" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.25"/>

<!-- match circles -->
<circle cx="480" cy="125" r="14" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.3" stroke-width="1.5"/>
<text x="480" y="130" font-size="13" text-anchor="middle" font-weight="500" fill="currentColor" fill-opacity="0.45">0</text>
<circle cx="480" cy="170" r="14" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.8"/>
<text x="480" y="175" font-size="13" text-anchor="middle" font-weight="600" fill="currentColor">1</text>
<circle cx="480" cy="215" r="14" fill="currentColor" fill-opacity="0.05" stroke="currentColor" stroke-opacity="0.3" stroke-width="1.5"/>
<text x="480" y="220" font-size="13" text-anchor="middle" font-weight="500" fill="currentColor" fill-opacity="0.45">0</text>

<!-- match -> values -->
<line x1="494" y1="125" x2="593" y2="125" stroke="#10B981" stroke-width="1.5" stroke-opacity="0.25"/>
<line x1="494" y1="170" x2="593" y2="170" stroke="#10B981" stroke-width="2.2"/>
<polygon points="600,170 593,166.5 593,173.5" fill="#10B981"/>
<line x1="494" y1="215" x2="593" y2="215" stroke="#10B981" stroke-width="1.5" stroke-opacity="0.25"/>

<!-- values -->
<rect x="600" y="109" width="80" height="32" rx="6" fill="#10B981" fill-opacity="0.05" stroke="#10B981" stroke-width="1.2" stroke-opacity="0.3"/>
<text x="640" y="130" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor" fill-opacity="0.4">1</text>
<rect x="600" y="154" width="80" height="32" rx="6" fill="#10B981" fill-opacity="0.16" stroke="#10B981" stroke-width="1.5"/>
<text x="640" y="175" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">2</text>
<rect x="600" y="199" width="80" height="32" rx="6" fill="#10B981" fill-opacity="0.05" stroke="#10B981" stroke-width="1.2" stroke-opacity="0.3"/>
<text x="640" y="220" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor" fill-opacity="0.4">3</text>

<!-- values -> result -->
<line x1="680" y1="170" x2="783" y2="170" stroke="#10B981" stroke-width="2.2"/>
<polygon points="790,170 783,166.5 783,173.5" fill="#10B981"/>
<rect x="790" y="154" width="100" height="32" rx="6" fill="#10B981" fill-opacity="0.16" stroke="#10B981" stroke-width="1.5"/>
<text x="840" y="175" font-size="14" text-anchor="middle" font-weight="600" fill="currentColor">2</text>

<!-- panel 1 notes -->
<text x="1080" y="125" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Exactly one key wins;</text>
<text x="1080" y="141" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">every other entry</text>
<text x="1080" y="157" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">contributes nothing</text>
<text x="1080" y="185" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">No match → KeyError</text>

<!-- ============ PANEL 2: soft dictionary ============ -->
<rect x="50" y="290" width="1010" height="250" rx="14" fill="currentColor" fill-opacity="0.025" stroke="currentColor" stroke-opacity="0.35" stroke-width="1.5"/>
<text x="70" y="314" font-size="14" text-anchor="start" font-weight="600" fill="currentColor">Soft dictionary: attention</text>
<text x="110" y="336" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">query</text>
<text x="305" y="336" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">keys</text>
<text x="480" y="336" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">weight = softmax(q · k)</text>
<text x="655" y="336" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">values</text>
<text x="890" y="336" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">result</text>

<!-- query vector -->
<rect x="75" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.25" stroke="#E8833A" stroke-width="1"/>
<rect x="89" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.55" stroke="#E8833A" stroke-width="1"/>
<rect x="103" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.35" stroke="#E8833A" stroke-width="1"/>
<rect x="117" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.7" stroke="#E8833A" stroke-width="1"/>
<rect x="131" y="429" width="14" height="22" fill="#E8833A" fill-opacity="0.3" stroke="#E8833A" stroke-width="1"/>

<!-- query -> keys (opacity tracks weight) -->
<path d="M150,440 H215 V385 H262" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-opacity="0.35" stroke-linejoin="round"/>
<polygon points="269,385 262,381.5 262,388.5" fill="#E8833A" fill-opacity="0.35"/>
<path d="M150,440 H215 V495 H262" fill="none" stroke="#E8833A" stroke-width="1.8" stroke-opacity="0.5" stroke-linejoin="round"/>
<polygon points="269,495 262,491.5 262,498.5" fill="#E8833A" fill-opacity="0.5"/>
<path d="M150,440 H262" fill="none" stroke="#E8833A" stroke-width="2.2" stroke-linejoin="round"/>
<polygon points="269,440 262,436.5 262,443.5" fill="#E8833A"/>

<!-- keys k1, k2, k3 -->
<rect x="270" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.45" stroke="#3B82F6" stroke-width="1"/>
<rect x="284" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.2" stroke="#3B82F6" stroke-width="1"/>
<rect x="298" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.3" stroke="#3B82F6" stroke-width="1"/>
<rect x="312" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.5" stroke="#3B82F6" stroke-width="1"/>
<rect x="326" y="374" width="14" height="22" fill="#3B82F6" fill-opacity="0.2" stroke="#3B82F6" stroke-width="1"/>
<text x="350" y="390" font-size="12" text-anchor="start" fill="currentColor">k₁</text>

<rect x="270" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.25" stroke="#3B82F6" stroke-width="1"/>
<rect x="284" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.55" stroke="#3B82F6" stroke-width="1"/>
<rect x="298" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.35" stroke="#3B82F6" stroke-width="1"/>
<rect x="312" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.65" stroke="#3B82F6" stroke-width="1"/>
<rect x="326" y="429" width="14" height="22" fill="#3B82F6" fill-opacity="0.3" stroke="#3B82F6" stroke-width="1"/>
<text x="350" y="445" font-size="12" text-anchor="start" fill="currentColor">k₂</text>

<rect x="270" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.6" stroke="#3B82F6" stroke-width="1"/>
<rect x="284" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.25" stroke="#3B82F6" stroke-width="1"/>
<rect x="298" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.5" stroke="#3B82F6" stroke-width="1"/>
<rect x="312" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.3" stroke="#3B82F6" stroke-width="1"/>
<rect x="326" y="484" width="14" height="22" fill="#3B82F6" fill-opacity="0.4" stroke="#3B82F6" stroke-width="1"/>
<text x="350" y="500" font-size="12" text-anchor="start" fill="currentColor">k₃</text>

<!-- keys -> weights -->
<line x1="378" y1="385" x2="420" y2="385" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.35"/>
<line x1="378" y1="440" x2="420" y2="440" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.6"/>
<line x1="378" y1="495" x2="420" y2="495" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.35"/>

<!-- weight bars (0.10, 0.70, 0.20) -->
<rect x="430" y="379" width="14" height="12" rx="2" fill="#E8833A" fill-opacity="0.9"/>
<text x="452" y="390" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.8">0.10</text>
<rect x="430" y="434" width="98" height="12" rx="2" fill="#E8833A" fill-opacity="0.9"/>
<text x="536" y="445" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.8">0.70</text>
<rect x="430" y="489" width="28" height="12" rx="2" fill="#E8833A" fill-opacity="0.9"/>
<text x="466" y="500" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.8">0.20</text>

<!-- weights -> values -->
<line x1="580" y1="385" x2="610" y2="385" stroke="#10B981" stroke-width="1.5" stroke-opacity="0.35"/>
<line x1="580" y1="440" x2="610" y2="440" stroke="#10B981" stroke-width="2.2"/>
<polygon points="617,440 610,436.5 610,443.5" fill="#10B981"/>
<line x1="580" y1="495" x2="610" y2="495" stroke="#10B981" stroke-width="1.5" stroke-opacity="0.45"/>

<!-- value vectors (opacity scaled by weight) -->
<rect x="620" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.12" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<rect x="634" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.2" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<rect x="648" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.1" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<rect x="662" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.18" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<rect x="676" y="374" width="14" height="22" fill="#10B981" fill-opacity="0.12" stroke="#10B981" stroke-width="1" stroke-opacity="0.4"/>
<text x="700" y="390" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.6">v₁</text>

<rect x="620" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.35" stroke="#10B981" stroke-width="1"/>
<rect x="634" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.7" stroke="#10B981" stroke-width="1"/>
<rect x="648" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.5" stroke="#10B981" stroke-width="1"/>
<rect x="662" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.3" stroke="#10B981" stroke-width="1"/>
<rect x="676" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.6" stroke="#10B981" stroke-width="1"/>
<text x="700" y="445" font-size="12" text-anchor="start" fill="currentColor">v₂</text>

<rect x="620" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.2" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<rect x="634" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.3" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<rect x="648" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.45" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<rect x="662" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.25" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<rect x="676" y="484" width="14" height="22" fill="#10B981" fill-opacity="0.35" stroke="#10B981" stroke-width="1" stroke-opacity="0.55"/>
<text x="700" y="500" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.8">v₃</text>

<!-- values -> sum -->
<path d="M726,385 H800 V432" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="800,433 796.5,426 803.5,426" fill="#10B981"/>
<line x1="726" y1="440" x2="791" y2="440" stroke="#10B981" stroke-width="2"/>
<path d="M726,495 H800 V448" fill="none" stroke="#10B981" stroke-width="2" stroke-linejoin="round"/>
<polygon points="800,447 796.5,454 803.5,454" fill="#10B981"/>
<circle cx="800" cy="440" r="8" fill="#10B981" fill-opacity="0.18" stroke="currentColor" stroke-opacity="0.7" stroke-width="1.5"/>
<line x1="795" y1="440" x2="805" y2="440" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>
<line x1="800" y1="435" x2="800" y2="445" stroke="currentColor" stroke-width="1.5" stroke-opacity="0.8"/>

<!-- sum -> result -->
<line x1="808" y1="440" x2="850" y2="440" stroke="#10B981" stroke-width="2.2"/>
<polygon points="857,440 850,436.5 850,443.5" fill="#10B981"/>
<rect x="860" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.3" stroke="#10B981" stroke-width="1"/>
<rect x="874" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.6" stroke="#10B981" stroke-width="1"/>
<rect x="888" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.45" stroke="#10B981" stroke-width="1"/>
<rect x="902" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.3" stroke="#10B981" stroke-width="1"/>
<rect x="916" y="429" width="14" height="22" fill="#10B981" fill-opacity="0.55" stroke="#10B981" stroke-width="1"/>
<text x="895" y="472" font-size="11" text-anchor="middle" fill="currentColor" fill-opacity="0.6" font-style="italic">weighted blend of values</text>

<!-- panel 2 notes -->
<text x="1080" y="385" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Every key matches a little</text>
<text x="1080" y="408" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Weights sum to 1</text>
<text x="1080" y="431" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Result is a blend, dominated</text>
<text x="1080" y="447" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">by the best-matching key</text>
<text x="1080" y="478" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">Smooth everywhere, so</text>
<text x="1080" y="494" font-size="12" text-anchor="start" fill="currentColor" fill-opacity="0.75">gradients can train it</text>

<!-- ============ LEGEND ============ -->
<rect x="60" y="570" width="34" height="14" rx="3" fill="#E8833A" fill-opacity="0.16" stroke="#E8833A" stroke-width="1.2"/>
<text x="104" y="581" font-size="12" text-anchor="start" fill="currentColor">Query: what we're looking up</text>
<rect x="60" y="592" width="34" height="14" rx="3" fill="#3B82F6" fill-opacity="0.16" stroke="#3B82F6" stroke-width="1.2"/>
<text x="104" y="603" font-size="12" text-anchor="start" fill="currentColor">Keys: the labels we match against</text>
<rect x="60" y="614" width="34" height="14" rx="3" fill="#10B981" fill-opacity="0.16" stroke="#10B981" stroke-width="1.2"/>
<text x="104" y="625" font-size="12" text-anchor="start" fill="currentColor">Values: the content we get back (shading shows how much of each is used)</text>
</svg>

In [2]:
'''
steps?

1. recreate a python dictionary using orthogonal vectors and softmax. This shows exactly how attention works as a dict.
  1.1. show naive softmax on 0/1 blurs the values together. Result: the final vector is a sum of the values, with the matched one weighted a bit more. Or equivalently: take the dot product of all values with the result to measure "how present" each one is.
  1.2. use high temperature in the softmax to account for this.
2. show one emergent generalization: we can recall the ideas about *nearly orthogonal* vectors from last week, and fit a lot more KV entries in the same-sized space.
  2.1. look at "how present" each possible value is as we add dims. If there's a clear difference, then attention is working b/c it can distinguish the chosen value from others.
  2.2. how does noise grow as we: change the dimensionality, change the number of entries, change the softmax temperature?
3. show another emergent generalization: we can let vectors be much closer *when being close is actually meaningful*. E.g., the embeddings of "cat" and "kitten" can be made close, so any value's we'd get for "cat," we'd get similarly for "kitten"
  3.1. The benefit is that a query near cat retrieves cat’s value. If cat and kitten are both stored with different values, a query near both returns a blend. That’s fuzzy matching.
4. (maybe) do backpropigation to show how it's differentiable, which is relevant to transformer training. (a few gradient descent steps on the keys to make a query retrieve a target value, showing the whole thing is differentiable)

note: possibly start with only 2 or 3 dims so we can visualize it. Then later go up to more
'''